In [1]:
import pandas as pd
import numpy as np

In [2]:
PATH_VAL_CLIM = 'Valores_Climatologicos_1970_2024_con_Coordenadas.csv'

In [3]:
COLS_NOMBRES = [
    'indicativo',
    'nombre',
    'provincia',
]

COLS_INTS = [
    'altitud'
]

COLS_FLOATS = [
    'tmed',
    'prec',
    'tmin',
    'tmax',
    'velmedia',
    'sol',
    'presMax',
    'presMin',
    'hrMedia',
    'dir',
    'racha',
    'hrMax',
    'hrMin'
]

COLS_NUMS = COLS_INTS + COLS_FLOATS

COL_FECHA = 'fecha'
COLS_HORAS = [
    'horaPresMax',
    'horaPresMin',
    'horatmin',
    'horatmax',
    'horaracha',
    'horaHrMax',
    'horaHrMin'
]

COLS_COORDS = [
    'lat',
    'lon'
]

In [4]:
df = pd.read_csv(PATH_VAL_CLIM,
                 sep=';', decimal = ',',
                 dtype = {nombre: 'string' for nombre in COLS_NOMBRES} |
                         {entero: 'Int64' for entero in COLS_INTS} |
                         {coordenada: 'Float64' for coordenada in COLS_COORDS} |
                         {otros: 'object' for otros in COLS_FLOATS + COLS_HORAS},
                         # {decimal: 'Float64' for decimal in COLS_FLOATS if decimal != 'prec'},
                 parse_dates = [COL_FECHA])

In [5]:
display(df)

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,tmax,velmedia,...,horatmax,dir,racha,horaracha,hrMax,horaHrMax,hrMin,horaHrMin,lon,lat
0,1970-01-01,C249I,FUERTEVENTURA AEROPUERTO,LAS PALMAS,25,"19,8","0,0","16,0","23,5","6,7",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-13.863056,28.444722
1,1970-01-01,1679A,MONFORTE DE LEMOS,LUGO,291,"4,0","0,0","0,0","8,0",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-7.510833,42.531667
2,1970-01-01,2462,PUERTO DE NAVACERRADA,MADRID,1893,"-5,0","0,4","-7,0","-3,0","3,1",...,15:30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.010556,40.793056
3,1970-01-01,1212E,ASTURIAS AEROPUERTO,ASTURIAS,127,"4,3","2,6","2,0","6,6","0,0",...,01:00,99.0,"7,2",00:13,NaN,NaN,NaN,NaN,-6.044167,43.566944
4,1970-01-01,0016A,REUS AEROPUERTO,TARRAGONA,71,"5,5","0,0","0,6","10,4","1,7",...,14:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.163611,41.145
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7427566,2024-12-31,9562X,MORELLA,CASTELLON,990,"4,8","0,0","0,8","8,8","1,4",...,13:42,12.0,"5,0",16:50,97.0,23:50,63.0,14:00,-0.101944,40.621667
7427567,2024-12-31,1002Y,"BAZTAN, IRURITA",NAVARRA,183,"4,2","0,0","-4,9","13,4","0,0",...,14:50,22.0,"3,1",16:20,100.0,09:00,50.0,14:10,-1.543056,43.135833
7427568,2024-12-31,3254Y,MORA,TOLEDO,717,"2,0","0,0","-3,1","7,1","1,1",...,11:18,20.0,"3,9",14:30,100.0,09:20,80.0,11:20,-3.780556,39.686944
7427569,2024-12-31,0194D,"CORBERA, PUIG D'AGULLES",BARCELONA,647,NaN,"0,0",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.885,41.408056


In [6]:
df_prep = df.copy(deep = True)

In [7]:
COL_PREC = 'prec'
# df_prep.loc[df[COL_PREC] == 'Ip', COL_PREC] = 0
df_prep = df_prep[df_prep[COL_PREC] != 'Acum']
df_prep[COL_PREC] = df_prep[COL_PREC].replace('Ip', 0)

In [8]:
df_prep[COLS_FLOATS] = df_prep[COLS_FLOATS].replace(',', '.', regex=True).apply(pd.to_numeric, errors = 'raise')

In [9]:
df_prep.dtypes

fecha          datetime64[ns]
indicativo     string[python]
nombre         string[python]
provincia      string[python]
altitud                 Int64
tmed                  float64
prec                  float64
tmin                  float64
tmax                  float64
velmedia              float64
sol                   float64
presMax               float64
horaPresMax            object
presMin               float64
horaPresMin            object
hrMedia               float64
horatmin               object
horatmax               object
dir                   float64
racha                 float64
horaracha              object
hrMax                 float64
horaHrMax              object
hrMin                 float64
horaHrMin              object
lon                   Float64
lat                   Float64
dtype: object

In [10]:
# PENDIENTE: LIMPIEZA Y CONVERSIÓN DE HORAS
# MIRAR DIR: 88 y 99

In [11]:
cols_horas = [
    'horaPresMax',
    'horaPresMin',
    'horatmin',
    'horatmax',
    'horaracha',
    'horaHrMax',
    'horaHrMin'
]

df_prep[cols_horas].astype(str).values

# sorted(list(set(df_prep[cols_horas].astype(str).values)))[-5:]

lista_vals_horas = df_prep[cols_horas].astype(str).values

lista_vals_horas_flat = [val for sublista in lista_vals_horas for val in sublista]

In [12]:
# Añadimos el sufijo :00 a las columnas cuyos valores carecen de él
cols_sufijo = ['horaPresMax', 'horaPresMin']
df_prep[cols_sufijo] = df_prep[cols_sufijo].where(
    df_prep[cols_sufijo].isna() | (df_prep[cols_sufijo] == 'Varias'),
    df_prep[cols_sufijo] + ':00'
)

In [13]:
# Procesamiento de horas
CATEGORIAS = ['Desconocida',
              'Varias',
              'Válida',
              'Inválida'] # Valor por defecto si no cuadra en ninguna de las demás

for col in cols_horas:

    filtro_formato = df_prep[col].str.match(r'^\d{1,2}:\d{2}$', na=False) # Comprueba que formato HH:MM
    df_hhmm = df_prep.loc[filtro_formato, col].str.split(":", expand=True).astype(int) # Df con dos columnas: HH y MM
    filtro_rango = (df_hhmm[0].between(0, 23)) & (df_hhmm[1].between(0, 59)) # Comprueba HH entre 0-23; MM entre 0-59 
    filtro_hhmm = filtro_rango.reindex(df_prep.index, fill_value = False) # Para mismo tamaño que original
    
    # Anotamos anomalías en la hora
    filtros = [
        df_prep[col].isna(),
        df_prep[col] == 'Varias',
        filtro_hhmm
    ]

    df_prep[col + '_tipo'] = np.select(filtros, CATEGORIAS[:-1], default = CATEGORIAS[-1])
    df_prep[col + '_tipo'] = df_prep[col + '_tipo'].astype('category')

    # Pasamos las horas a tipo time
    df_prep[col] = pd.to_datetime(
        df_prep[col], format = '%H:%M', errors = 'coerce'
    ).dt.time

    # Creamos columna datetime respectiva
    df_prep['fecha' + col] = pd.to_datetime(
        df_prep[COL_FECHA].dt.date.astype(str) + ' ' + df_prep[col].astype(str),
        format = '%Y-%m-%d %H:%M',
        errors = 'coerce'
    )
    

In [14]:
# Procesamos dirección del viento
dict_dir = {
    88: 'Desconocida',
    99: 'Varias'
}
dir_por_defecto = 'Válida'

df_prep['dir_tipo'] = df_prep['dir'].map(dict_dir).fillna(dir_por_defecto)

In [15]:
df_prep.to_csv('Valores_Climatologicos_1970_2024_Limpios.csv', sep = ';', index = False)